In [1]:
import numpy as np
import matplotlib.pyplot as plt
import imageio
from matplotlib.colors import ListedColormap

In [ ]:
def game_theory_sim(b, duration, make_gif=False, gen_random=False):

    SIZE = 201

    Cooperator = np.int8(0)
    Defector = np.int8(1)

    if not gen_random:
        # 1 for cooperator, -1 for defector
        matrix = np.full((SIZE, SIZE), Cooperator)
        # start with one defector in the middle
        matrix[100, 100] = Defector
    else:
        matrix = np.random.randint(0, 2, (SIZE, SIZE))


    strategy = np.array([
        [1,0],
        [b,0]
    ])

    if make_gif:
        frames = []
        colors = ['blue', 'red', 'yellow', 'green']
        cmap = ListedColormap(colors)

    for _ in range(duration):
        right = np.roll(matrix, 1, axis=1)
        left = np.roll(matrix, -1, axis=1)
        up = np.roll(matrix, 1, axis=0)
        down = np.roll(matrix, -1, axis=0)

        up_right = np.roll(right, 1, axis=0)
        up_left = np.roll(left, 1, axis=0)
        down_right = np.roll(right, -1, axis=0)
        down_left = np.roll(left, -1, axis=0)

        score = strategy[matrix, matrix] + strategy[matrix, right] + strategy[matrix, left] + strategy[matrix, up] + strategy[matrix, down] + strategy[matrix, up_left] + strategy[matrix, up_right] + strategy[matrix, down_left] + strategy[matrix, down_right]
        
        score_right = np.roll(score, 1, axis=1)
        score_left = np.roll(score, -1, axis=1)
        score_up = np.roll(score, 1, axis=0)
        score_down = np.roll(score, -1, axis=0)
        score_up_right = np.roll(score_right, 1, axis=0)
        score_up_left = np.roll(score_left, 1, axis=0)
        score_down_right = np.roll(score_right, -1, axis=0)
        score_down_left = np.roll(score_left, -1, axis=0)

        all_scores = np.stack([
            score,
            score_right,
            score_left,
            score_up,
            score_down,
            score_up_left,
            score_up_right,
            score_down_left,
            score_down_right
        ])

        strategy_neighbors = np.stack([
            matrix,
            right,
            left,
            up,
            down,
            up_left,
            up_right,
            down_left,
            down_right
        ])

        best_score_id = np.argmax(all_scores, axis=0) 
        # axis = 0 because all scores is (9, size, size)

        old_matrix = matrix
        matrix = np.take_along_axis(
            strategy_neighbors,
            best_score_id[np.newaxis,:,:],
            axis=0
        )[0]

        if make_gif:
            visual_matrix = np.zeros((SIZE, SIZE), dtype=int)
            visual_matrix[(old_matrix == 0) & (matrix == 0)] = 0 # C->C
            visual_matrix[(old_matrix == 0) & (matrix == 1)] = 1 # D->C
            visual_matrix[(old_matrix == 1) & (matrix == 0)] = 2 # C->D
            visual_matrix[(old_matrix == 1) & (matrix == 1)] = 3 # D->D

            frame_img = (cmap(visual_matrix / 3.0)[:, :, :3] * 255).astype(np.uint8)
            frames.append(frame_img)

    if make_gif:
        if gen_random:
            imageio.mimsave(f'game_theory_random_{b}.gif', frames, fps=10, loop=0)
        else:
            imageio.mimsave(f'game_theory_{b}.gif', frames, fps=10, loop=0)
        print("gif saved!")

    return np.sum(matrix) / SIZE**2 

print(game_theory_sim(1.9, 150, True))
print(game_theory_sim(2.08, 150, True))

In [ ]:
start = 1.0
stop = 2.5
count = 50
duration_per_run = 500
b_values = np.linspace(start, stop, count)

results = []
i = 0
for b in b_values:
    
    print(f"b = {round(b, 3)}")
    make_gif = False
    if i % 5 == 0:
        make_gif=True
        
    f = game_theory_sim(b, duration_per_run, make_gif=make_gif, gen_random=True)
    # print(f"-> f = {f}")
    results.append((b,f))
    i += 1

results = np.array(results)
print(results.shape)

plt.scatter(results[:,0], results[:,1])
plt.xlabel("b")
plt.ylabel("f")

In [ ]:
def game_theory_sim_extended(b, duration, make_gif=False):
    SIZE = 201

    # Actions
    C = np.int8(0)
    D = np.int8(1)

    # Strategies
    ALLC = np.int8(0)
    ALLD = np.int8(1)
    TFT = np.int8(2)
    PAVLOV = np.int8(3)

    N_STRATEGIES = 4

    # Random initial distribution
    strategy_matrix = np.random.randint(0, N_STRATEGIES, (SIZE, SIZE), dtype=np.int8)

    # Define the 9 interaction directions:
    # 0:self, 1:Right, 2:Left, 3:Down, 4:Up, 5:Down-Right, 6:Up-Left, 7:Down-Left, 8:Up-Right
    dr  = np.array([0,  0,  0,  1, -1,  1, -1,  1, -1])
    dc  = np.array([0,  1, -1,  0,  0,  1, -1, -1,  1])
    # The opposite directions (to fetch what the opponent played against US)
    opp = np.array([0,  2,  1,  4,  3,  6,  5,  8,  7])

    # Initial last moves: Shape is now (9, SIZE, SIZE) to store a distinct move for every neighbor
    last_move = np.random.randint(0, 2, (9, SIZE, SIZE), dtype=np.int8)

    # Payoff matrix [my_move, opp_move] -> [Row, Col]
    payoff = np.array([
        [1.0, 0.0],
        [b,   0.0]
    ])

    # Visualisation setup
    if make_gif:
        colors = ['blue', 'red', 'green', 'yellow']
        cmap = ListedColormap(colors)
        frames = []

    # MAIN LOOP
    for _ in range(duration):
        current_move = np.zeros((9, SIZE, SIZE), dtype=np.int8)
        last_opponent_move = np.zeros((9, SIZE, SIZE), dtype=np.int8)

        # 1. Determine what each opponent played against us last round
        for d in range(9):
            # We look at the opposite direction matrix, shifted back to our perspective
            last_opponent_move[d] = np.roll(last_move[opp[d]], shift=(-dr[d], -dc[d]), axis=(0, 1))

        # 2. Determine our current moves for all 9 interactions based on strategy
        for d in range(9):
            # ALLC
            current_move[d][strategy_matrix == ALLC] = C
            # ALLD
            current_move[d][strategy_matrix == ALLD] = D
            # TFT (Play what they played against us last time)
            current_move[d][strategy_matrix == TFT] = last_opponent_move[d][strategy_matrix == TFT]
            
            # Pavlov (Win-stay, Lose-shift. Cooperate if both did the same move last round)
            pav_mask = (strategy_matrix == PAVLOV)
            current_move[d][pav_mask] = (last_move[d][pav_mask] != last_opponent_move[d][pav_mask]).astype(np.int8)

        # 3. Determine what opponents are playing against us THIS round to score
        current_opponent_move = np.zeros((9, SIZE, SIZE), dtype=np.int8)
        for d in range(9):
            current_opponent_move[d] = np.roll(current_move[opp[d]], shift=(-dr[d], -dc[d]), axis=(0, 1))

        # 4. Score calculation
        score = np.zeros((SIZE, SIZE), dtype=float)
        for d in range(9):
            # Advanced indexing matches our move vs their move directly to the payoff matrix
            score += payoff[current_move[d], current_opponent_move[d]]

        # 5. Find the best neighbor strategy
        neighbor_scores = np.zeros((9, SIZE, SIZE), dtype=float)
        neighbor_strategies = np.zeros((9, SIZE, SIZE), dtype=np.int8)
        
        for d in range(9):
            neighbor_scores[d] = np.roll(score, shift=(-dr[d], -dc[d]), axis=(0, 1))
            neighbor_strategies[d] = np.roll(strategy_matrix, shift=(-dr[d], -dc[d]), axis=(0, 1))

        best_idx = np.argmax(neighbor_scores, axis=0)

        # Update strategy matrix and squeeze the 3D output back to 2D
        strategy_matrix = np.take_along_axis(
            neighbor_strategies,
            best_idx[np.newaxis, :, :],
            axis=0
        )[0]
        
        # Save memory for next round
        last_move = current_move

        # frame gen
        if make_gif:
            frame_img = (
                cmap(strategy_matrix / (N_STRATEGIES - 1))[:, :, :3] * 255
            ).astype(np.uint8)
            frames.append(frame_img)

    if make_gif:
        imageio.mimsave(
            f'game_theory_multistrategy_b{b}.gif',
            frames,
            fps=4,
            loop=0
        )
        print(f"gif saved! -> game_theory_multistrategy_b{b}.gif")

    return strategy_matrix

game_theory_sim_extended(2.1, 50, True)

gif saved! -> game_theory_multistrategy_b2.1.gif


array([[2, 2, 2, ..., 2, 2, 2],
       [2, 2, 2, ..., 2, 2, 2],
       [2, 2, 2, ..., 2, 2, 2],
       ...,
       [2, 2, 2, ..., 2, 2, 2],
       [2, 2, 2, ..., 2, 1, 1],
       [2, 2, 2, ..., 1, 1, 1]], shape=(201, 201), dtype=int8)